[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/31_gradient_accumulation_solution.ipynb)

# ✅ Solution: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss`jax.grad``
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import math


In [ ]:
# ✅ SOLUTION

import jax
from flax import nnx
def accumulated_step(model,optimizer,loss_fn,micro_batches):
    optimizer.zero_grad(); n=len(micro_batches); total=0.
    def loss(m,x,y): return loss_fn(m(x),y)
    for x,y in micro_batches:
        value,grads=nnx.value_and_grad(loss)(model,x,y); total+=float(value)/n
        grads=jax.tree.map(lambda g:g/n,grads)
        optimizer.grads=grads if optimizer.grads is None else jax.tree.map(lambda a,b:a+b,optimizer.grads,grads)
    optimizer.step(); return total


In [ ]:
# Verify
print(accumulated_step)


In [ ]:
from jax_judge import check
check("gradient_accumulation")
